# Tokenizers

[Back to the introduction](ch01_demo.ipynb). Run in order in a fresh kernel. `LLM.tokenizer()` loads only a model's tokenizer files, not its weights, so this notebook downloads a few MB and needs no inference. Loading is async, but `encode()` and `decode()` are plain synchronous calls. See [Tokenize-only handles](https://github.com/decitre/lmlite/blob/main/docs/USAGE.md#tokenize-only-handles) for details.

In [ ]:
from lmlite import LLM

Three tokenizers with different designs: GPT-2 (byte-level BPE), multilingual BERT (WordPiece), and POTION, a static embedding model. POTION has no transformer: a sentence vector is the mean of rows looked up in a table by these token ids.

In [ ]:
tokenizers = {
    "gpt2": await LLM.tokenizer(model="gpt2"),
    "bert-multilingual": await LLM.tokenizer(model="Xenova/bert-base-multilingual-cased"),
    "potion": await LLM.tokenizer(model="minishlab/potion-base-8M"),
}
for name, tok in tokenizers.items():
    print(f"{name}: vocabulary of {tok.vocab_size} tokens")

`encode()` returns a list of ints. Decoding each id on its own shows how each tokenizer splits the same text: each piece is printed above its id.

In [ ]:
text = "Tokenizers split café menus into pieces."
for name, tok in tokenizers.items():
    ids = tok.encode(text)
    pieces = [repr(tok.decode([i])) for i in ids]  # quoted, so leading spaces show
    widths = [max(len(piece), len(str(i))) for piece, i in zip(pieces, ids)]
    print(f"{name:18} " + " ".join(piece.ljust(w) for piece, w in zip(pieces, widths)))
    print(f"{'':18} " + " ".join(str(i).ljust(w) for i, w in zip(ids, widths)))

Special tokens are off by default. `add_special_tokens=True` adds the markers a model expects around its input, such as BERT's `[CLS]` and `[SEP]`. GPT-2 adds none.

In [ ]:
for name, tok in tokenizers.items():
    print(f"{name:18} {tok.decode(tok.encode(text, add_special_tokens=True))}")

Since `encode()` needs no `await`, it works inside ordinary functions, for example to check that a prompt fits a model's context before generating. The same sentence costs a different number of tokens in each language and tokenizer.

In [ ]:
def token_count(tok, text):
    return len(tok.encode(text))  # synchronous: no await


sentences = {
    "English": "The pharmacy is right next to the sushi bar.",
    "French": "La pharmacie est juste à côté du bar à sushis.",
    "Japanese": "薬局は寿司屋のすぐ隣にあります。",
    "Russian": "Аптека находится рядом с суши-баром.",
}
for language, sentence in sentences.items():
    counts = {name: token_count(tok, sentence) for name, tok in tokenizers.items()}
    print(f"{language:9} {counts}")

Decoding the ids gives the text back, after each tokenizer's normalization: POTION lowercases and strips accents, for example.

In [ ]:
for name, tok in tokenizers.items():
    print(f"{name:18} {tok.decode(tok.encode(text))}")

In [ ]:
for tok in tokenizers.values():
    tok.close()

Closed tokenizers raise on further use. Tokenizer files only go through the browser's HTTP cache, so `cached_models()` does not list them.